In [1]:
!pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.5 MB/s eta 0:00:00


In [2]:
import os
import gc
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG (Tối ưu cho mit_b3)
# ==========================================
ENCODER_NAME = 'mit_b3' 
DATA_PATH = '/kaggle/input/datasets/phantunhngtom/danang/Data_Training_Soft_NPZ'

PATCH_SIZE = 256
BATCH_SIZE = 12      # mit_b3 nặng hơn, để 12 là an toàn cho 16GB VRAM
NUM_WORKERS = 4      
LEARNING_RATE = 3e-4 # Transformer thích LR này
EPOCHS = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tăng tốc tính toán
torch.backends.cudnn.benchmark = True

# ==========================================
# 2. DATASET (Giữ nguyên logic tối ưu I/O)
# ==========================================
class KaggleFloodDataset(Dataset):
    def __init__(self, folder_path, files, g_min, g_max, patch_size=256):
        self.folder_path = folder_path
        self.patch_size = patch_size
        self.file_list = files
        self.g_min = torch.tensor(g_min).float().view(-1, 1, 1)
        self.g_max = torch.tensor(g_max).float().view(-1, 1, 1)
        self.patches_info = []
        self.last_file = None
        self.last_data = None

        print(f">>> Indexing {len(files)} files...")
        def process_file(f_name):
            valid_patches = []
            try:
                data = np.load(os.path.join(self.folder_path, f_name))
                y_full = data['y']
                h, w = y_full.shape
                for i in range(0, h - self.patch_size + 1, self.patch_size):
                    for j in range(0, w - self.patch_size + 1, self.patch_size):
                        if np.any(y_full[i:i+patch_size, j:j+patch_size] > 0.001):
                            valid_patches.append((f_name, i, j))
            except: pass # Bỏ qua file lỗi
            return valid_patches

        with ThreadPoolExecutor(max_workers=4) as executor:
            results = list(tqdm(executor.map(process_file, self.file_list), total=len(self.file_list)))
            for res in results: self.patches_info.extend(res)
        
        print(f">>> Found {len(self.patches_info)} valid patches.")

    def __len__(self): return len(self.patches_info)

    def __getitem__(self, idx):
        f_name, h_start, w_start = self.patches_info[idx]
        if self.last_file == f_name:
            data = self.last_data
        else:
            data = np.load(os.path.join(self.folder_path, f_name), mmap_mode='r')
            self.last_file, self.last_data = f_name, data
        
        x = torch.from_numpy(data['x'][:, h_start:h_start+self.patch_size, w_start:w_start+self.patch_size].copy()).float()
        y = torch.from_numpy(data['y'][h_start:h_start+self.patch_size, w_start:w_start+self.patch_size].copy()).float().unsqueeze(0)
        x = (x - self.g_min) / (self.g_max - self.g_min + 1e-7)
        return x, y

# ==========================================
# 3. MÔ HÌNH (Tối ưu cho MiT-Transformer)
# ==========================================
class ASPP(nn.Module):
    def __init__(self, in_dims, out_dims):
        super().__init__()
        self.conv1 = nn.Conv2d(in_dims, out_dims, 1, bias=False)
        self.conv2 = nn.Conv2d(in_dims, out_dims, 3, padding=6, dilation=6, bias=False)
        self.conv3 = nn.Conv2d(in_dims, out_dims, 3, padding=12, dilation=12, bias=False)
        self.conv_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_dims, out_dims, 1, bias=False))
        self.fuse = nn.Sequential(nn.Conv2d(out_dims * 4, out_dims, 1), nn.BatchNorm2d(out_dims), nn.ReLU(inplace=True))

    def forward(self, x):
        h, w = x.shape[-2:]
        res = [self.conv1(x), self.conv2(x), self.conv3(x), 
               F.interpolate(self.conv_pool(x), size=(h, w), mode='bilinear', align_corners=True)]
        return self.fuse(torch.cat(res, dim=1))

class FlexibleFloodModel(nn.Module):
    def __init__(self, encoder_name, n_channels=8):
        super().__init__()
        # mit_b3 không hỗ trợ decoder_attention_type='scse' tốt như CNN trong một số bản SMP, 
        # nên ta dùng bản tiêu chuẩn để đảm bảo ổn định.
        self.base = smp.Unet(
            encoder_name=encoder_name, encoder_weights="imagenet",
            in_channels=n_channels, classes=1
        )
        # Lấy số kênh cuối của mit_b3 (thường là 512)
        final_channels = self.base.encoder.out_channels[-1]
        self.aspp = ASPP(final_channels, final_channels)

    def forward(self, x):
        features = self.base.encoder(x)
        features[-1] = self.aspp(features[-1])
        decoder_output = self.base.decoder(features)
        return self.base.segmentation_head(decoder_output)

# ==========================================
# 4. LOSS (Giữ nguyên logic thủy văn)
# ==========================================
class AdvancedHydroLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, pred_logits, target, inputs):
        mask = ((target > 0.001) & (target < 0.999)).float()
        bce = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
        flow_w = 1.0 + torch.clamp(inputs[:, 7:8] * 12.0, 0, 15)
        loss_bce = (bce * mask * flow_w).sum() / (mask.sum() + 1e-7)
        probs = torch.sigmoid(pred_logits)
        inter = (probs * target * mask).sum()
        union = (probs * mask).sum() + (target * mask).sum()
        loss_dice = 1 - (2. * inter + 1e-7) / (union + 1e-7)
        return 0.4 * loss_bce + 0.6 * loss_dice

# ==========================================
# 5. TRAINING PIPELINE
# ==========================================
def train():
    all_files = sorted([f for f in os.listdir(DATA_PATH) if f.endswith('.npz')])
    random.seed(42); random.shuffle(all_files)
    
    # Split 70/15/15
    n = len(all_files)
    train_files = all_files[:int(n*0.7)]
    val_files = all_files[int(n*0.7):int(n*0.85)]
    test_files = all_files[int(n*0.85):]

    # Lưu Split ra phần Output
    with open("dataset_split.json", "w") as f:
        json.dump({"train": train_files, "val": val_files, "test": test_files}, f)

    # Tính Stats nhanh
    mins, maxs = [], []
    for f in random.sample(train_files, min(20, len(train_files))):
        d = np.load(os.path.join(DATA_PATH, f))['x']
        mins.append(d.min(axis=(1,2))); maxs.append(d.max(axis=(1,2)))
    g_min = np.array(mins).min(axis=0); g_max = np.array(maxs).max(axis=0)

    train_ds = KaggleFloodDataset(DATA_PATH, train_files, g_min, g_max)
    val_ds = KaggleFloodDataset(DATA_PATH, val_files, g_min, g_max)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, 
                            pin_memory=True, persistent_workers=True)

    model = FlexibleFloodModel(ENCODER_NAME).to(DEVICE)
    # Mit_b3 hoạt động tốt nhất với định dạng mặc định hoặc channels_last
    model = model.to(memory_format=torch.channels_last)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE*3, 
                                              steps_per_epoch=len(train_loader), epochs=EPOCHS)
    criterion = AdvancedHydroLoss()
    scaler = GradScaler()

    best_loss = float('inf')
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
        
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): 
                pred = model(x)
                loss = criterion(pred, y, x)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Val Phase
        model.eval()
        val_l = 0
        with torch.no_grad():
            for vx, vy in val_loader:
                vx = vx.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
                vy = vy.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    val_l += criterion(model(vx), vy, vx).item()
        
        avg_v_loss = val_l / len(val_loader)
        print(f"Epoch {epoch} | Val Loss: {avg_v_loss:.4f}")
        
        if avg_v_loss < best_loss:
            best_loss = avg_v_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'g_min': g_min, 'g_max': g_max, 'files': {'test': test_files}
            }, f"best_model_{ENCODER_NAME}.pth")
            print(">>> Saved Best!")

        gc.collect()
        torch.cuda.empty_cache()

if __name__ == "__main__":
    train()

>>> Indexing 393 files...


  0%|          | 0/393 [00:00<?, ?it/s]

>>> Found 9210 valid patches.
>>> Indexing 84 files...


  0%|          | 0/84 [00:00<?, ?it/s]

>>> Found 1910 valid patches.


...

Epoch 33 | Val Loss: 0.5137


Epoch 34/35:   0%|          | 0/768 [00:00<?, ?it/s]

Epoch 34 | Val Loss: 0.5133


Epoch 35/35:   0%|          | 0/768 [00:00<?, ?it/s]

Epoch 35 | Val Loss: 0.5136
